In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
sys.path.append('../src')

import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import umap


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

In [23]:
from data_loading import *
from eda import *
from data_cleaning import *
from k_means import *
from cluster_evaluation import *
from cluster_interpretation import *

In [16]:
embeddings_numpy_arr = np.load("../data/analysis_data/embeddings.npy")
embeddings_df = pd.DataFrame(embeddings_numpy_arr)

In [17]:
path = "../data/analysis_data/only_english.csv"
only_english = read_file(path, seperator=",")
# only_english = only_english[['product_name', 'categories_tags', 'nova_group', 'ingredients_text', 'ingredient_count', 'additives_n', 'nutriscore_score', 'monounsaturated-fat_100g','sugars_100g','polyunsaturated-fat_100g','sugar_fiber_ratio','carbohydrates_100g','fiber_100g','saturated-fat_100g','sodium_per_calorie']]
print(only_english.shape)
only_english.head()

Shape of the data: (272809, 18)

Columns loaded:
['code', 'product_name', 'categories_tags', 'ingredients_text', 'ingredient_count', 'additives_n', 'additive_density', 'nutriscore_score', 'monounsaturated-fat_100g', 'sugars_100g', 'polyunsaturated-fat_100g', 'sugar_fiber_ratio', 'carbohydrates_100g', 'fiber_100g', 'saturated-fat_100g', 'sodium_per_calorie', 'allergens_en', 'nova_group']
(272809, 18)


,code,product_name,categories_tags,ingredients_text,ingredient_count,additives_n,additive_density,nutriscore_score,monounsaturated-fat_100g,sugars_100g,polyunsaturated-fat_100g,sugar_fiber_ratio,carbohydrates_100g,fiber_100g,saturated-fat_100g,sodium_per_calorie,allergens_en,nova_group
0,00000028,Hershey’s Syrup,"en:beverages-and-beverages-preparations,en:bev...",organ defat hemp seed powder certifi organ qua...,1,0.0,0.000000,NaN,0.0,6.0,0.0,0.594059,15.00,10.000,0.0,NaN,NaN,1
1,00000116,Vanille-Erdbeer-Cocktail,NaN,sunflow seed butter roast sunflow kernel dri c...,30,4.0,0.129032,NaN,0.0,0.0,0.0,NaN,8.82,0.441,0.0,NaN,NaN,4
2,00000226,Naturelo,NaN,methylcellulos color preserv flavong ount per ...,5,1.0,0.166667,NaN,0.0,0.0,0.0,NaN,0.00,0.000,0.0,NaN,NaN,4
3,00000261,Prebiotic Fiber Gummies,"en:snacks,en:sweet-snacks,en:confectioneries,e...",chicori root fiber water pectin agar agar tapi...,11,6.0,0.500000,NaN,0.0,0.0,0.0,NaN,0.00,0.000,0.0,NaN,NaN,4
4,00000283,Irish Sea Moss,NaN,irish sea moss extract chondru crispu hypromel...,3,2.0,0.500000,NaN,0.0,0.0,0.0,NaN,0.00,0.000,0.0,NaN,NaN,4


In [18]:
cluster_df = merge_data_frames(df1=only_english, df2 = embeddings_df)
cluster_df.head()

,code,product_name,categories_tags,ingredients_text,ingredient_count,additives_n,additive_density,nutriscore_score,monounsaturated-fat_100g,sugars_100g,...,374,375,376,377,378,379,380,381,382,383
0,00000028,Hershey’s Syrup,"en:beverages-and-beverages-preparations,en:bev...",organ defat hemp seed powder certifi organ qua...,1,0.0,0.000000,NaN,0.0,6.0,...,-0.021125,0.020245,0.064699,-0.034334,0.051692,-0.022212,0.006391,0.025445,0.077194,-0.048097
1,00000116,Vanille-Erdbeer-Cocktail,NaN,sunflow seed butter roast sunflow kernel dri c...,30,4.0,0.129032,NaN,0.0,0.0,...,-0.002987,0.008882,0.038631,-0.054230,0.064427,-0.033297,-0.031501,-0.038130,-0.006603,-0.042481
2,00000226,Naturelo,NaN,methylcellulos color preserv flavong ount per ...,5,1.0,0.166667,NaN,0.0,0.0,...,-0.073252,0.015455,0.074647,0.074792,0.046393,-0.074582,-0.020494,-0.082925,0.045365,-0.041050
3,00000261,Prebiotic Fiber Gummies,"en:snacks,en:sweet-snacks,en:confectioneries,e...",chicori root fiber water pectin agar agar tapi...,11,6.0,0.500000,NaN,0.0,0.0,...,0.057201,0.003271,0.063289,0.023518,0.026960,0.024571,-0.015965,-0.011356,0.067122,-0.004868
4,00000283,Irish Sea Moss,NaN,irish sea moss extract chondru crispu hypromel...,3,2.0,0.500000,NaN,0.0,0.0,...,-0.035997,0.041002,0.092975,-0.031187,0.005940,-0.003002,-0.091133,-0.081211,0.068467,0.013320


In [19]:
TEXT_CANDS = ["ingredients_text","ingredients_text_en","ingredients_text_with_allergens"]
NUTRI_CANDS = ["energy-kcal_100g","energy_100g","proteins_100g","carbohydrates_100g",
               "sugars_100g","fiber_100g","fat_100g","saturated-fat_100g","salt_100g","sodium_100g"]
META_CANDS  = ["code","product_name","brands","categories","compared_to_category","countries_tags"]
LABEL_CANDS = ["nutriscore_grade","nutriscore_score","nova_group","nova_groups_tags",
               "additives_n","additives_tags","allergens","allergens_tags"]

In [20]:
text_col = next((c for c in TEXT_CANDS if c in cluster_df.columns), None)
nutri_cols = [c for c in NUTRI_CANDS if c in cluster_df.columns]
meta_cols  = [c for c in META_CANDS  if c in cluster_df.columns]
label_cols = [c for c in LABEL_CANDS if c in cluster_df.columns]

usecols = sorted(set(nutri_cols + meta_cols + label_cols + ([text_col] if text_col else [])))
print("Using columns (first 25):", usecols[:25], "… total:", len(usecols))

Using columns (first 25): ['additives_n', 'carbohydrates_100g', 'code', 'fiber_100g', 'ingredients_text', 'nova_group', 'nutriscore_score', 'product_name', 'saturated-fat_100g', 'sugars_100g'] … total: 10


In [ ]:
df = cluster_df.copy()


In [30]:
num_cols = [c for c in NUTRI_CANDS if c in df.columns]
# flag_cols = [c for c in df.columns if c.startswith("allergen_")] + [c for c in ["additives_n","sweetener_any_flag","sweetener_his_flag","sweetener_polyol_flag"] if c in df.columns]
id_cols = [c for c in ["code","product_name","brands","categories","compared_to_category","countries_tags"] if c in df.columns]
text_col = text_col if text_col in df.columns else None

In [31]:
cluster_df.columns.tolist()

['code',
 'product_name',
 'categories_tags',
 'ingredients_text',
 'ingredient_count',
 'additives_n',
 'additive_density',
 'nutriscore_score',
 'monounsaturated-fat_100g',
 'sugars_100g',
 'polyunsaturated-fat_100g',
 'sugar_fiber_ratio',
 'carbohydrates_100g',
 'fiber_100g',
 'saturated-fat_100g',
 'sodium_per_calorie',
 'allergens_en',
 'nova_group',
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 1

In [32]:
flag_cols = ['allergens_en']

In [33]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
flag_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent"))])
txt_pipe  = Pipeline([("tfidf", TfidfVectorizer(min_df=5, max_df=0.98, ngram_range=(1,2),
                                                token_pattern=r"(?u)\b[a-z0-9][a-z0-9\-]+\b",
                                                max_features=10000)),
                      ("svd", TruncatedSVD(n_components=100, random_state=42))])

feature_sets = {}

if len(num_cols) >= 4:
    feature_sets["A_num"] = ColumnTransformer([("num", num_pipe, num_cols)], remainder="drop")

if len(num_cols) >= 4 and len(flag_cols) >= 1:
    feature_sets["B_num_flags"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                     ("flg", flag_pipe, flag_cols)], remainder="drop")

if text_col:
    if len(num_cols) >= 4:
        feature_sets["C_num_text"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                        ("txt", txt_pipe, text_col)], remainder="drop")
    else:
        feature_sets["C_text"] = ColumnTransformer([("txt", txt_pipe, text_col)], remainder="drop")

if text_col and len(flag_cols) >= 1:
    if len(num_cols) >= 4:
        feature_sets["D_num_flags_text"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                              ("flg", flag_pipe, flag_cols),
                                                              ("txt", txt_pipe, text_col)], remainder="drop")
    else:
        feature_sets["D_flags_text"] = ColumnTransformer([("flg", flag_pipe, flag_cols),
                                                          ("txt", txt_pipe, text_col)], remainder="drop")

list(feature_sets.keys())


['A_num', 'B_num_flags', 'C_num_text', 'D_num_flags_text']

In [34]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import numpy as np

def eval_scores(X, labels, cap=4000):
    idx = np.arange(X.shape[0])
    if len(idx) > cap:
        idx = np.random.choice(idx, cap, replace=False)
    return (
        float(silhouette_score(X[idx], labels[idx])),
        float(calinski_harabasz_score(X[idx], labels[idx])),
        float(davies_bouldin_score(X[idx], labels[idx]))
    )

def sweep_kmeans(X, ks):
    out = []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        lab = km.fit_predict(X)
        sil, ch, db = eval_scores(X, lab)
        out.append(("kmeans", k, sil, ch, db, lab, km))
    return out

def sweep_gmm(X, ks):
    out = []
    for k in ks:
        gm = GaussianMixture(n_components=k, covariance_type="full", random_state=42)
        gm.fit(X)
        lab = gm.predict(X)
        sil, ch, db = eval_scores(X, lab)
        out.append(("gmm", k, sil, ch, db, lab, gm))
    return out


In [35]:
results, embeds = [], {}
k_grid_km = [4,6,8,10]
k_grid_gmm = [4,6,8]

for name, ct in feature_sets.items():
    X = ct.fit_transform(df)
    embeds[name] = X
    for algo,k,sil,ch,db,lab,model in (sweep_kmeans(X, k_grid_km) + sweep_gmm(X, k_grid_gmm)):
        results.append({"feat":name,"algo":algo,"k":k,
                        "silhouette":sil,"calinski_harabasz":ch,"davies_bouldin":db,
                        "labels":lab,"model":model})

res_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ("labels","model")} for r in results])
res_df = res_df.sort_values("silhouette", ascending=False)
res_df.head(10)


TypeError: Feature names are only supported if all input features have string names, but your input has ['int', 'str'] as feature name / column name types. If you want feature names to be stored and validated, you must convert them all to strings, by using X.columns = X.columns.astype(str) for example. Otherwise you can remove feature / column names from your input data, or convert them all to a non-string data type.

In [ ]:

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
flag_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent"))])
txt_pipe  = Pipeline([("tfidf", TfidfVectorizer(min_df=5, max_df=0.98, ngram_range=(1,2),
                                                token_pattern=r"(?u)\b[a-z0-9][a-z0-9\-]+\b",
                                                max_features=10000)),
                      ("svd", TruncatedSVD(n_components=100, random_state=42))])

feature_sets = {}

if len(num_cols) >= 4:
    feature_sets["A_num"] = ColumnTransformer([("num", num_pipe, num_cols)], remainder="drop")

if len(num_cols) >= 4 and len(flag_cols) >= 1:
    feature_sets["B_num_flags"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                     ("flg", flag_pipe, flag_cols)], remainder="drop")

if text_col:
    if len(num_cols) >= 4:
        feature_sets["C_num_text"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                        ("txt", txt_pipe, text_col)], remainder="drop")
    else:
        feature_sets["C_text"] = ColumnTransformer([("txt", txt_pipe, text_col)], remainder="drop")

if text_col and len(flag_cols) >= 1:
    if len(num_cols) >= 4:
        feature_sets["D_num_flags_text"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                              ("flg", flag_pipe, flag_cols),
                                                              ("txt", txt_pipe, text_col)], remainder="drop")
    else:
        feature_sets["D_flags_text"] = ColumnTransformer([("flg", flag_pipe, flag_cols),
                                                          ("txt", txt_pipe, text_col)], remainder="drop")

list(feature_sets.keys())


In [ ]:
cluster_df.columns.tolist()
df = cluster_df[]

['code',
 'product_name',
 'categories_tags',
 'ingredients_text',
 'ingredient_count',
 'additives_n',
 'additive_density',
 'nutriscore_score',
 'monounsaturated-fat_100g',
 'sugars_100g',
 'polyunsaturated-fat_100g',
 'sugar_fiber_ratio',
 'carbohydrates_100g',
 'fiber_100g',
 'saturated-fat_100g',
 'sodium_per_calorie',
 'allergens_en',
 'nova_group',
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 1